In [2]:


import os
import re
import pandas as pd
import numpy as np
import torch
from datasets import load_dataset


from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# OpenRouter
from openai import OpenAI

# UI
import gradio as gr
from dotenv import load_dotenv
load_dotenv(override=True)




True

### LOad Documents 

In [3]:
DATA_PATH = "data"

documents = []

for file in os.listdir(DATA_PATH):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(DATA_PATH, file))
        documents.extend(loader.load())

print("Total pages loaded:", len(documents))

Total pages loaded: 429


### Add Metadata (Very Important)

This allows the AI to cite page numbers and document names.

In [4]:
for doc in documents:
    doc.metadata["source"] = doc.metadata.get("source", "unknown")

### Split Documents into Chunks

This improves retrieval accuracy.

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 1352


### Create Embeddings

We will use embeddings from Hugging Face.

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Create Vector Database

Using Chroma.

In [7]:
from langchain_community.vectorstores import Chroma

vector_db = Chroma.from_documents(
    chunks,
    embedding_model,
    persist_directory="legal_vector_db"
)

vector_db.persist()

/var/folders/6q/bvcphsk90d5f5r13tvp9kt480000gn/T/ipykernel_6660/2749423511.py:9: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vector_db.persist()


### Retrieval Function

In [8]:
def retrieve_context(question):

    docs = vector_db.similarity_search(question, k=4)

    context = ""
    sources = []

    for doc in docs:
        context += doc.page_content + "\n\n"

        source = f"{doc.metadata.get('source')} | page {doc.metadata.get('page')}"
        sources.append(source)

    return context, sources

### Connect to OpenRouter

We call an LLM through OpenRouter.

In [9]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

In [10]:
def ask_llm(question):

    context, sources = retrieve_context(question)

    prompt = f"""
You are an expert Nigerian electoral law assistant.

Use ONLY the context provided to answer.

If the answer is not in the context, say:
"I could not find this in the electoral documents."

Context:
{context}

Question:
{question}
"""

    response = client.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    answer = response.choices[0].message.content

    sources_text = "\n".join(set(sources))

    return f"{answer}\n\nSources:\n{sources_text}"

In [11]:
chat_history = []

In [12]:
def chat(question):

    response = ask_llm(question)

    chat_history.append((question, response))

    return response

In [13]:
import gradio as gr

interface = gr.Interface(
    fn=chat,
    inputs=gr.Textbox(
        lines=3,
        placeholder="Ask a question about electoral law..."
    ),
    outputs="text",
    title="Nigerian Electoral Law AI Assistant",
    description="Ask questions about the Electoral Act and Constitution"
)

interface.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
